In [1]:
!pip install -q pyspark

In [2]:
import urllib.request
from pyspark.sql import SparkSession

# SparkSession 생성
spark = SparkSession.builder \
    .appName("Q2_IntPair") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
# 입력 파일 다운로드
url = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/int-pair-input.txt"
urllib.request.urlretrieve(url, "int-pair-input.txt")

# Spark RDD로 파일 읽기
lines = sc.textFile("int-pair-input.txt")

In [5]:
# 빈 줄을 제외하고, 문자열 '(x, y)'를 정수 쌍 [x, y]로 변환 (Transformation)
pairs_rdd = (lines
    .filter(lambda line: line.strip() != "")
    .map(lambda line: [int(v) for v in line.replace("(", "")
                                               .replace(")", "")
                                               .replace(",", " ").split()]))

# 모든 정수 쌍에서 첫 번째 원소만 추출 (Transformation)
first_elems_rdd = pairs_rdd.map(lambda pair: pair[0])

# 수업에서 배운 Action을 이용해 필요한 숫자만 Driver로 반환
# RDD 전체를 collect()하여 Python for문으로 처리하지 않음
v1 = pairs_rdd.first()[0]                             # 첫 번째 정수 쌍의 첫 원소
v2 = first_elems_rdd.reduce(lambda a, b: max(a, b))   # 최댓값
v3 = first_elems_rdd.reduce(lambda a, b: min(a, b))   # 최솟값
v4 = first_elems_rdd.filter(lambda x: x == 1).count() # 첫 원소가 1인 쌍의 개수
v5 = first_elems_rdd.filter(lambda x: x == 2).count() # 첫 원소가 2인 쌍의 개수

# 다섯 개의 계산 결과를 RDD로 생성
results_rdd = sc.parallelize([v1, v2, v3, v4, v5])

# Spark reduce Action을 사용하여 최종 합계 계산
final_answer = results_rdd.reduce(lambda a, b: a + b)

print(f"1. 첫 번째 정수 쌍의 첫 원소: {v1}")
print(f"2. 첫 번째 원소 최댓값: {v2}")
print(f"3. 첫 번째 원소 최솟값: {v3}")
print(f"4. 첫 번째 원소가 1인 개수: {v4}")
print(f"5. 첫 번째 원소가 2인 개수: {v5}")
print(f"\n최종 합계: {final_answer}")

1. 첫 번째 정수 쌍의 첫 원소: 2
2. 첫 번째 원소 최댓값: 32
3. 첫 번째 원소 최솟값: 0
4. 첫 번째 원소가 1인 개수: 31
5. 첫 번째 원소가 2인 개수: 28

최종 합계: 93
